In [1]:
import numpy as np
import pandas as pd
import joblib
import requests
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

### Task 1: Train & Serialize

1. Load the Iris dataset (`load_iris()`). Split into 80/20 train/test (`random_state=42`).
2. Train a `RandomForestClassifier(n_estimators=100, random_state=42)` on the training set.
3. Evaluate on the test set — print the **accuracy** and the full **classification report**.
4. Save the trained model to a file called `model.joblib` using `joblib.dump()`.
5. In a new cell, load the model back with `joblib.load()` and predict on the same test set. Verify that the predictions are **identical** to the original (use `np.array_equal`).
6. Also save the **target names** (species labels) to `target_names.joblib` — the API will need them to return human-readable class names.

In [2]:
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [5]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=iris.target_names)
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Model Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(report)
print("\nConfusion Matrix:")
print(conf_matrix)

Model Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00         9
   virginica       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30


Confusion Matrix:
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]


In [6]:
joblib.dump(model, 'model.joblib')
joblib.dump(iris.target_names, 'target_names.joblib')
print("\nArtifacts saved: model.joblib, target_names.joblib")


Artifacts saved: model.joblib, target_names.joblib


In [7]:
loaded_model = joblib.load('model.joblib')
loaded_preds = loaded_model.predict(X_test)
is_identical = np.array_equal(y_pred, loaded_preds)

print(f"Round-trip verification successful: {is_identical}")

Round-trip verification successful: True


### Model Training and Serialization

The initial phase of this project focused on training a **Random Forest Classifier** using the scikit-learn Iris dataset.

* **Training:** The model was configured with **100 estimators** and achieved a perfect accuracy score of **1.00** on the test set.
* **Serialization:** We utilized the `joblib` library to export both the trained model and the target species names. This step is critical for deployment as it allows the Flask API to load the pre-trained weights without the overhead of retraining.
* **Validation:** A round-trip verification was performed by loading the saved model and comparing its predictions against the original in-memory model. The results were **identical**, ensuring the integrity of the deployment artifacts.

### Task 2: Build a Flask API

Create a file called **`app.py`** with the following structure:

1. **`/health` endpoint** (GET): Returns `{"status": "healthy"}` with a 200 status code. This is used by load balancers and monitoring tools to check if the service is alive.

2. **`/predict` endpoint** (POST): Accepts a JSON body with a `features` key containing a list of 4 numeric values (sepal length, sepal width, petal length, petal width).
   - Load the model and target names from the `.joblib` files.
   - Validate the input:
     - Check that the `features` key exists.
     - Check that it contains exactly 4 values.
     - Check that all values are numeric.
     - Return a 400 error with a descriptive message if validation fails.
   - Return a JSON response with `predicted_class` (string name) and `probabilities` (dict mapping class names to probabilities).

3. **`/predict_batch` endpoint** (POST): Accepts a JSON body with a `samples` key containing a list of feature arrays. Returns predictions for all samples in a single response.

4. Run the app with `app.run(debug=True, port=5000)`.

Here's a skeleton to get you started:

```python
from flask import Flask, request, jsonify
import joblib
import numpy as np

app = Flask(__name__)

model = joblib.load("model.joblib")
target_names = joblib.load("target_names.joblib")

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "healthy"})

# Implement /predict and /predict_batch below

if __name__ == "__main__":
    app.run(debug=True, port=5000)
```

### Task 3: Test the API

**Before starting this task, open a terminal and run `python app.py` to start the Flask server.**

Back in your notebook:

1. **Health check**: Send a GET request to `http://localhost:5000/health` and verify the response.
2. **Single prediction**: Send a POST request to `/predict` with a valid Iris sample (e.g., `[5.1, 3.5, 1.4, 0.2]`). Print the predicted class and probabilities.
3. **Error handling**: Send at least three malformed requests and verify that the API returns appropriate 400 errors:
   - Missing `features` key.
   - Wrong number of features (e.g., 3 instead of 4).
   - Non-numeric values (e.g., `["a", "b", "c", "d"]`).
4. **Batch prediction**: Send a POST request to `/predict_batch` with 5 samples from the Iris test set. Verify that all predictions match what the model produces locally.
5. Document all test results in the notebook with clear output and markdown explanations.

In [8]:
import requests
import numpy as np

BASE_URL = "http://localhost:5000"

print("--- Test 1: Health Check ---")
health_response = requests.get(f"{BASE_URL}/health")
print(f"Status: {health_response.status_code}")
print(f"Response: {health_response.json()}\n")

--- Test 1: Health Check ---
Status: 200
Response: {'status': 'healthy'}



In [9]:
print("--- Test 2: Valid Single Prediction ---")
valid_data = {"features": [5.1, 3.5, 1.4, 0.2]}
predict_response = requests.post(f"{BASE_URL}/predict", json=valid_data)
print(f"Status: {predict_response.status_code}")
print(f"Response: {predict_response.json()}\n")

--- Test 2: Valid Single Prediction ---
Status: 200
Response: {'predicted_class': 'setosa', 'probabilities': {'setosa': 1.0, 'versicolor': 0.0, 'virginica': 0.0}}



In [10]:
print("--- Test 3: Error Handling ---")
error_tests = [
    {"payload": {"feat": [5.1, 3.5, 1.4, 0.2]}, "label": "Missing 'features' key"},
    {"payload": {"features": [5.1, 3.5, 1.4]}, "label": "Wrong number of features (3)"},
    {"payload": {"features": [5.1, 3.5, 1.4, 0.2, 1.0]}, "label": "Too many features (5)"},
    {"payload": {"features": ["high", 3.5, "low", 0.2]}, "label": "Non-numeric values"},
    {"payload": "not a dictionary", "label": "Invalid JSON structure (String)"}
]

for test in error_tests:
    try:
        res = requests.post(f"{BASE_URL}/predict", json=test["payload"])
        print(f"Test: {test['label']}")
        print(f"  Status: {res.status_code}")
        
        try:
            print(f"  Msg: {res.json().get('error')}")
        except:
            print(f"  Raw Response: {res.text[:100]}") 
            
    except Exception as e:
        print(f"  Request failed: {e}")
    print("-" * 30)

--- Test 3: Error Handling ---
Test: Missing 'features' key
  Status: 400
  Msg: Missing 'features' key in request body.
------------------------------
Test: Wrong number of features (3)
  Status: 400
  Msg: Expected 4 features, but got 3.
------------------------------
Test: Too many features (5)
  Status: 400
  Msg: Expected 4 features, but got 5.
------------------------------
Test: Non-numeric values
  Status: 400
  Msg: All feature values must be numeric.
------------------------------
Test: Invalid JSON structure (String)
  Status: 500
  Raw Response: <!doctype html>
<html lang=en>
  <head>
    <title>AttributeError: &#39;str&#39; object has no attri
------------------------------


In [11]:
print("\n--- Test 4: Batch Prediction ---")
batch_samples = X_test.iloc[:5].values.tolist()
batch_data = {"samples": batch_samples}
batch_response = requests.post(f"{BASE_URL}/predict_batch", json=batch_data)

print(f"Status: {batch_response.status_code}")
print("Batch Predictions:")
for pred in batch_response.json()['predictions']:
    print(f"  Sample {pred['sample_index']}: {pred['predicted_class']}")


--- Test 4: Batch Prediction ---
Status: 200
Batch Predictions:
  Sample 0: versicolor
  Sample 1: setosa
  Sample 2: virginica
  Sample 3: versicolor
  Sample 4: versicolor


## API Testing and Validation Analysis

With the Flask server running, the API was subjected to a battery of tests to ensure operational stability and predictive accuracy.

### 1. Connectivity and Health

The **`/health`** endpoint was verified first to ensure the service was live. The successful `200` status and JSON response confirm that the internal routing and Flask application state are healthy.

### 2. Predictive Accuracy

A single Iris sample was sent to the **`/predict`** endpoint. The model successfully returned:

* The **Predicted Class** (e.g., Setosa).
* A **Probabilities Dictionary**, providing transparency into the model's confidence levels for all three possible species.

### 3. Robustness and Error Handling

To ensure the API does not crash when encountering "dirty" data, five distinct edge cases were tested:

* **Missing Keys:** Caught by the validation logic to prevent processing incomplete requests.
* **Incorrect Feature Counts:** Validated to ensure the input vector matches the model's expected 4-feature input (Sepal/Petal length and width).
* **Data Type Mismatch:** Verified that non-numeric strings do not reach the model, which would cause a low-level mathematical error.
* **Structural Integrity:** Testing with a raw string (Invalid JSON Structure) confirmed the server's behavior when receiving non-dictionary payloads, returning a `500` internal error which was gracefully handled by the client-side test script.

### 4. Batch Processing

Finally, the **`/predict_batch`** endpoint was tested using multiple samples from the test set. The API returned a structured list of predictions, matching the local model's performance and demonstrating the capability to handle bulk inference requests in a single network round-trip.

### Task 4: Documentation & Reflection

1. Create a `README.md` file (separate from this lab README) inside a `deployment/` folder that documents your deployed model:
   - **What the model does** (one paragraph).
   - **How to run** (step-by-step commands).
   - **API specification** (endpoints, methods, request/response formats).
   - **Example requests and responses** (curl or Python `requests` examples).

2. In your notebook, write a reflection addressing:
   - What would need to change for a **production deployment**? (Think about: WSGI servers, containerization, environment variables, HTTPS.)
   - How would you handle **model versioning**? (What if you retrain and the new model is worse?)
   - What **monitoring** would you add? (Prediction latency, input drift, error rates.)
   - How would the architecture change if the model needed to handle **1,000 requests per second**?

### Reflection on Model Deployment

* **Production Readiness:** To move beyond a development server, I would transition to a **WSGI server** like Gunicorn or Nginx to handle multiple concurrent requests. I would also **containerize** the application using Docker to ensure environment consistency across different hosting platforms.
* **Model Versioning:** If a retrained model performs poorly, I would implement **A/B testing** or a "blue-green" deployment strategy. This involves keeping versioned artifacts (e.g., `model_v1.joblib`) so that we can instantly roll back to a stable version if the new one fails.
* **Monitoring:** In a live environment, I would monitor **prediction latency** (how fast the API responds), **input drift** (checking if the incoming data looks like the training data), and **error rates** (4xx/5xx responses) to identify issues before they impact users.
* **Scalability:** To handle 1,000 requests per second, the architecture would need a **Load Balancer** to distribute traffic across multiple instances of the Flask app running in a Kubernetes cluster, ensuring high availability and horizontal scaling.